# Chapter 13 on Amazon Neptune

Run in a Neptune workbench notebook (graph-notebook). **This creates billable resources: follow `TEARDOWN.md` when you finish.**

First load the tiny graph used throughout the book (Alice knows Bob and Dave; both know Carol).

In [ ]:
%%oc
CREATE (a:Person {name:'Alice'}), (b:Person {name:'Bob'}),
       (c:Person {name:'Carol'}), (d:Person {name:'Dave'}),
       (a)-[:KNOWS]->(b), (b)-[:KNOWS]->(c),
       (a)-[:KNOWS]->(d), (d)-[:KNOWS]->(c)

## Listing 13.1 — The same question in openCypher, Gremlin, and SPARQL on Neptune

openCypher and Gremlin read the same property-graph data. SPARQL reads RDF, which is a separate data model in Neptune, so load the triples first (next cell).

In [ ]:
%%oc
MATCH (:Person {name:'Alice'})-[:KNOWS]->(f:Person) RETURN f.name AS friend

In [ ]:
%%gremlin
g.V().has('Person','name','Alice').out('knows').values('name')

In [ ]:
%%sparql
PREFIX ex: <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
INSERT DATA {
  ex:alice foaf:name "Alice" ; foaf:knows ex:bob , ex:dave .
  ex:bob foaf:name "Bob" ; foaf:knows ex:carol .
  ex:dave foaf:name "Dave" ; foaf:knows ex:carol .
  ex:carol foaf:name "Carol" .
}

In [ ]:
%%sparql
PREFIX ex: <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name WHERE { ex:alice foaf:knows ?f . ?f foaf:name ?name . }

## Listing 13.2 — Running an algorithm on Neptune Analytics (openCypher)

Run this against a **Neptune Analytics** graph (not Neptune Database).

In [ ]:
%%oc
// PageRank over the whole in-memory graph, stored on each node as 'rank'
CALL neptune.algo.pageRank.mutate({writeProperty: 'rank'})
YIELD success
RETURN success

## Bulk load from S3 (Section 13.5)

Upload `bulk-load/vertices.csv` and `bulk-load/edges.csv` to an S3 bucket in the same region, give the cluster an IAM role that can read it, then run the loader. Replace the bucket, role ARN, and region.

In [ ]:
%load -s s3://YOUR-BUCKET/graph-mastery/ -f csv --iam-role arn:aws:iam::ACCOUNT:role/NeptuneLoadFromS3 --region us-east-1

In [ ]:
%%gremlin
g.V().hasLabel('Person').count()